In [63]:
import sys
sys.path.append("../preprocess")
import glob
from qbiolip import load_db
import re
import torch
df = load_db("../../databases/QBioLiP/qbiolip.csv", "../../databases/ec_uniprot_ids.txt")
df = df.loc[df.target]

result_path = "/home/daniil/Desktop/Ph.D/DynamicBind/results/*/*"
result_folders = glob.glob(result_path)
result_folders.sort()

targets = []
preds = []
names = []
def xyz_from_sdf(sdf_p):
    with open(sdf_p) as sdf_f:
        lines = [l.split() for l in sdf_f]
    lines = [l for l in lines if len(l) == 16] #only xyz coordinates lines
    xyz = torch.tensor([
        [
            float(l[0]), #x
            float(l[1]), #y
            float(l[2]), #z
        ]
    for l in lines])
    return xyz 

for folder in result_folders:
    if "affinity_prediction.csv" in folder:
        continue
    cases = glob.glob(f"{folder}/*")
    target = xyz_from_sdf(f"{folder}/ref_ligandFile.sdf")
    # target_file = [c for c in cases if re.search(r'')]
    protein_name = [c.split("/")[-1].replace(".pdb", "") for c in cases if re.search(r'^[0-9a-z]{4}_[0-9A-Z_]*.pdb', c.split("/")[-1]) is not None][0]
    ligand_name = [c.split("/")[-1].replace(".sdf", "") for c in cases if re.search(r'^[0-9a-z]{4}_[0-9A-Z_]*.sdf', c.split("/")[-1]) is not None][0]
    name = ligand_name if ligand_name != protein_name else f"{ligand_name}_trimed"
    print(name)
    pred_sdfs = [c for c in cases if re.search(r"rank[0-9]{1,2}_ligand_[a-z_0-9.]*_relaxed.sdf", c.split("/")[-1]) is not None]
    if len(pred_sdfs) == 0:
        continue
    pred = torch.stack([xyz_from_sdf(p) for p in pred_sdfs])
    ranks = torch.tensor([int(re.findall(r'(?<=rank)[0-9]{1,2}(?=_)', c)[0]) for c in pred_sdfs])
    ranks_sorted = ranks.argsort(descending=False)
    pred = pred[ranks_sorted]

    targets.append(target)
    preds.append(pred)
    names.append(name)

print(len(targets))
print(len(preds))
print(len(names))

Len of the database: 25415
Columns: id, pdb_id, assembly_id, stoichiometry, resolution, uniprot_id, assembly_detail, ligand_id, site, ligand_detail, pubmed_id, interaction, relevant, binding_site, binding_site_pdb, bindingMOAD, pdbbind_cn, binding_db, group, lig_canon_smiles, lig_atoms, rec_len_resi, centroid_x, centroid_y, centroid_z, x_size, y_size, z_size, target, exp_type, size_x, size_y, size_z
1b4n_1_GUA_G
1b4n_1_GUA_G_trimed
1b4n_1_GUA_K
1b4n_1_GUA_K_trimed
1b8a_1_ATP_J
1b8a_1_ATP_J_trimed
1b9z_1_GLC_B
1b9z_1_GLC_B_trimed
1c0t_1_BM1_C
1c0t_1_BM1_C_trimed
1f48_1_ADP_Q
1f48_1_ADP_Q_trimed
1f48_2_ADP_R3
1f48_2_ADP_R3_trimed
1f4l_1_MET_C2
1f4l_1_MET_C2_trimed
1ffy_1_MRC_P
1ffy_1_MRC_P_trimed
1gl9_2_ANP_E
1gl9_2_ANP_E_trimed
1obc_1_LEU_B
1obc_1_LEU_B_trimed
1oyy_1_AGS_E
1oyy_1_AGS_E_trimed
1pj9_1_GLC_G
1pj9_1_GLC_G_trimed
1qf6_1_AMP_D
1qf6_1_AMP_D_trimed
1qha_1_GLC_E
1qha_1_GLC_E_trimed
1qpb_1_PYM_H1
1qpb_1_PYM_H1_trimed
1qvr_1_ANP_M
1qvr_1_ANP_M_trimed
1r31_3_MEV_G1
1r31_3_MEV_G1_tr